In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

# If this is the primary file that is executed (ie not an import of another file)
if __name__ == "__main__":
    # Download the dataset if not already present
    !wget -nc https://raw.githubusercontent.com/AnirudhKondapally/NLP-Sentiment-Analysis-RNN-CNN-LSTM/master/dataset/amazon_cells_labelled.txt

    # get data, pre-process and split
    data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index                                          # add new column index
    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)                             # pre-process

    # Split the data into training, validation sets, preserving raw text for LSTM and labels
    raw_training_sentences, raw_validation_sentences, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
        data['Sentence'].values.astype('U'),
        data['Class'].values.astype('int32'),
        test_size=0.10,
        random_state=0,
        shuffle=True
    )
    # Save a copy of raw validation sentences specifically for LSTM
    val_texts_for_lstm = raw_validation_sentences.copy()

    # VEKTORIZÁLÁS - Ez alakítja át a szöveges adatot mátrixszá (az ANN-nek)
    # Initialize TFIDF vectorizer
    word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')

    # Apply TFIDF to the raw text sentences to get vectorized data for the ANN
    training_data_tfidf = word_vectorizer.fit_transform(raw_training_sentences)        # transform texts to sparse matrix
    training_data_tfidf = training_data_tfidf.todense()                             # convert to dense matrix for Pytorch
    vocab_size = len(word_vectorizer.vocabulary_)

    validation_data_tfidf = word_vectorizer.transform(raw_validation_sentences)
    validation_data_tfidf = validation_data_tfidf.todense()

    train_x_tensor = torch.from_numpy(np.array(training_data_tfidf)).type(torch.FloatTensor)
    train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
    validation_x_tensor = torch.from_numpy(np.array(validation_data_tfidf)).type(torch.FloatTensor)
    validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()

    class Net(nn.Module):
      def __init__ (self,sz):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(sz,64)
        self.fc2 = nn.Linear(64,64)
        self.fc3 = nn.Linear(64,1)

      def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return torch.sigmoid(x)

model = Net(vocab_size)
print(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

tset = TensorDataset(train_x_tensor, train_y_tensor.float().unsqueeze(1))
tloader = DataLoader(tset, batch_size=32, shuffle=True)

for epoch in range(10):
  model.train()
  running_loss = 0.0
  for i,(inputs,labels) in enumerate(tloader):
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()
        running_loss += loss.item()

  print(f"epoch {epoch + 1} avg loss: {running_loss / len(tloader):.4f}")



<>:29: SyntaxWarning: invalid escape sequence '\.'
<>:30: SyntaxWarning: invalid escape sequence '\w'
<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:29: SyntaxWarning: invalid escape sequence '\.'
<>:30: SyntaxWarning: invalid escape sequence '\w'
<>:31: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_10815/1802428208.py:29: SyntaxWarning: invalid escape sequence '\.'
  data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
/tmp/ipykernel_10815/1802428208.py:30: SyntaxWarning: invalid escape sequence '\w'
  data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
/tmp/ipykernel_10815/1802428208.py:31: SyntaxWarning: invalid escape sequence '\d'
  data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
[nltk_data] Downloading pa

File ‘amazon_cells_labelled.txt’ already there; not retrieving.



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Net(
  (fc1): Linear(in_features=7277, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)
epoch 1 avg loss: 0.6913
epoch 2 avg loss: 0.6411
epoch 3 avg loss: 0.4458
epoch 4 avg loss: 0.1760
epoch 5 avg loss: 0.0499
epoch 6 avg loss: 0.0179
epoch 7 avg loss: 0.0097
epoch 8 avg loss: 0.0064
epoch 9 avg loss: 0.0042
epoch 10 avg loss: 0.0031


In [2]:
import torch.nn.utils.rnn as rnn_utils


all_words = " ".join(data['Sentence']).split()
vocab = {word: i+1 for i, word in enumerate(set(all_words))}
vocab_size = len(vocab) + 1

def sentence_to_indices(sentence, vocab):
    return [vocab[w] for w in sentence.split() if w in vocab]


encoded_sentences = [sentence_to_indices(s, vocab) for s in data['Sentence']]

max_len = 20
padded_sentences = [s[:max_len] + [0]*(max_len - len(s)) for s in encoded_sentences]


train_x_lstm = torch.LongTensor(padded_sentences)
train_y_lstm = torch.from_numpy(data['Class'].values).float().unsqueeze(1)

class LSTMNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMNet, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embedded)

        last_hidden = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)

        out = self.fc(last_hidden)
        return self.sigmoid(out)

model_lstm = LSTMNet(vocab_size, embedding_dim=64, hidden_dim=32)
print(model_lstm)

LSTMNet(
  (embedding): Embedding(2469, 64)
  (lstm): LSTM(64, 32, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [3]:
lstm_dataset = TensorDataset(train_x_lstm, train_y_lstm)
lstm_loader = DataLoader(lstm_dataset, batch_size=32, shuffle=True)

criterion_lstm = nn.BCELoss()
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=0.001)

for epoch in range(5):
    model_lstm.train()
    total_loss = 0
    for batch_x, batch_y in lstm_loader:
        optimizer_lstm.zero_grad()
        outputs = model_lstm(batch_x)
        loss = criterion_lstm(outputs, batch_y)
        loss.backward()
        optimizer_lstm.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(lstm_loader):.4f}")

Epoch 1, Loss: 0.6869
Epoch 2, Loss: 0.6664
Epoch 3, Loss: 0.6264
Epoch 4, Loss: 0.5613
Epoch 5, Loss: 0.4813


In [5]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
transformer_results = classifier(list(val_texts_for_lstm))
trans_preds = [1 if res['label'] == 'POSITIVE' else 0 for res in transformer_results]
trans_acc = accuracy_score(validation_labels, trans_preds)
print(f"Transformer Accuracy: {trans_acc * 100:.2f}%")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Transformer Accuracy: 92.00%


In [6]:
model_lstm.eval()
with torch.no_grad():
    val_encoded = [sentence_to_indices(s, vocab) for s in val_texts_for_lstm]
    val_padded = [s[:max_len] + [0]*(max_len - len(s)) for s in val_encoded]
    val_x_lstm_tensor = torch.LongTensor(val_padded)

    lstm_out = model_lstm(val_x_lstm_tensor)
    lstm_preds = (lstm_out > 0.5).float().flatten()

    correct = (lstm_preds == torch.tensor(validation_labels).float()).sum().item()
    print(f"LSTM Accuracy: {correct / len(validation_labels) * 100:.2f}%")

LSTM Accuracy: 87.00%


In [7]:
model.eval()
with torch.no_grad():
    ann_out = model(validation_x_tensor)
    ann_preds = (ann_out > 0.5).float().flatten()
    ann_acc = (ann_preds == validation_y_tensor.float()).float().mean()
    print(f"Simple ANN Accuracy: {ann_acc.item() * 100:.2f}%")

Simple ANN Accuracy: 85.00%


Conlusion:
The transformer gave us the best result followed by LSTM which was 87% then the Simple ANN whith 85% accuracy.
If speed is the priority we would recommend Simple ANN but only for low-rescue environment. If we need to train a model without using massive pre-trained weights we would choose LSTM. Any other case Transformer is really good beacuse of its superior understanding of context.

Complexity: Transformer is the most complex, then LSTM and the simplest is Simple ANN

Accuracy: ANN does not remember the order of words so it makes it less accurate but LSTM knows. Transformer uses a whole sentece at once that makes it really accurate.

Efficiency: Simple ANN has the fastest training time.

Data:
ANN and LSTM needs a lot of data to learn from zero. Transformer is more data-efficient it was already pre-trained on a lot of words.

Embeddings:
Embeddings represent words in a mathematical space where words with similar words are close to each other but TF-IDF only count words.

Architecture:
LSTM can handle the sequences and Transformer's attention are the key reasons they outperformed the basic linear network.


